In [0]:
class Silver_results():
    main_path="/Volumes/formula1_race/default/formula1/"
    bronze_path = "formula1_race_project/bronze"
    silver_path = "formula1_race_project/silver"

    def __init__(self,folder):
         self.folder_name=folder
        
    def read_input(self):
        from pyspark.sql.functions import col,max,count
        if spark.catalog.tableExists("formula1_race.silver.results"):
            max_ingestion_date=spark.read.table('formula1_race.silver.results').agg(max(col('results_ingestion_date')).alias('max_ingestion_date')).collect()[0]['max_ingestion_date']
            if max_ingestion_date is None:
                max_ingestion_date='1900-01-01 00:00:00'
            print("if_max_ingestion_date:",max_ingestion_date)
            print(f"if_max_ingestion_date:",type(max_ingestion_date))
            read_df=spark.read.table('formula1_race.bronze.results').filter(col('ResultsIngestionDate')>max_ingestion_date)
    
            print("reading results read_df")
            display(read_df.select(count(col('resultId'))))
        else:
            max_ingestion_date='1900-01-01 00:00:00'
            print("else_max_ingestion_date:",max_ingestion_date)
            read_df=spark.read.table('formula1_race.bronze.results').filter(col('ResultsIngestionDate')>max_ingestion_date)
        return read_df
    
    def column_name_formating(self,read_df):
        colrename_df=read_df
        import re
        for c in colrename_df.columns:
            result = re.sub(r'([a-z])([A-Z])',r'\1,\2',c).lower().split(',')
            new_column="_".join(result)
            colrename_df=colrename_df.withColumnRenamed(c,new_column)
        return colrename_df
    
    def apply_transformations(self,colrename_df):
        from pyspark.sql.functions import round,col,when
        apply_tran_df=colrename_df
        for c,t in apply_tran_df.dtypes:
            if t=='string':
                print(c)
                apply_tran_df=apply_tran_df.withColumn(c,when(col(c).isin('\\N'),'-').otherwise(col(c)))
            elif t in ['int', 'bigint']:
                print(c)
                apply_tran_df=apply_tran_df.withColumn(c,when(col(c).isNull(),0000).otherwise(col(c)))
            elif t in ['float', 'double']:
                print(c)
                apply_tran_df=apply_tran_df.withColumn(c,when(col(c).isNull(),0.0).otherwise(col(c)))
                
        apply_tran_df= (apply_tran_df.selectExpr("result_id","race_id","driver_id","constructor_id",
                                                "number","grid","position as result_position","position_text as result_position_text","position_order as result_position_order",
                                                "points as result_points","laps","time",
                                                "milliseconds","fastest_lap","rank as fastest_lap_rank",
                                                "fastest_lap_time","fastest_lap_speed",
                                                "status_id","results_ingestion_date","source")
                        )
        
        return apply_tran_df
    
    def write_output(self,apply_tran_df):
        apply_tran_df.write.partitionBy("race_id").mode("append").saveAsTable("formula1_race.silver.results")
        display(spark.sql("select count(*) from formula1_race.silver.results"))
        print("Data write into sliver results table is Done")
        
       
     
    def process(self):
        print("Started silver-ingestion-results  in ran....")
        read_df=self.read_input()
        colrename_df=self.column_name_formating(read_df)
        apply_tran_df=self.apply_transformations(colrename_df)
        self.write_output(apply_tran_df)

In [0]:
Silver_results_instance = Silver_results("results")
Silver_results_instance.process()